In [42]:
######IMPORT LIBRARIES #######
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import ipywidgets as widgets

from IPython.display import display, clear_output

In [43]:
#####WORK ON EXCEL DATA #####

file_path = r"C:\Users\husnu\Downloads\23_Personal-Budget-Dahboard-Template-TemplateLab.com_.xlsx"

raw_df = pd.read_excel(file_path, sheet_name="Data", header=None)

print("Raw data shape:", raw_df.shape)
display(raw_df.head(12))

Raw data shape: (116, 10)


,0,1,2,3,4,5,6,7,8,9
0,Credit Card info,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Card Holder,Card Number,Valid Thru,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,JOHN DOE,**** **** **** 0000,11/26,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Starting Balance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1000,NaN,NaN,NaN,DATE,TRANSACTION,FROM/TO,AMOUNT,NOTE,BALANCE
6,NaN,NaN,NaN,NaN,2023-01-01 00:00:00,Income,MyCompany,4200,Salary - December 2022,5200
7,NaN,NaN,NaN,NaN,2023-01-02 00:00:00,Housing,Vavilon LTD,-105,Home maintenance,5095
8,Transaction Categories,NaN,Selected Month,NaN,2023-01-03 00:00:00,Transportation,Parkings,-10,Parking,5085
9,Income,NaN,2023-01-01 00:00:00,NaN,2023-01-04 00:00:00,Food,McDonalds,-40,NaN,5045


In [44]:
######EXTTRACT THE ACTUAL TRANSACTION TABLE #####

# Actual header row index
header_row = 5

# Extract only transaction table columns
df = raw_df.iloc[header_row + 1:, 4:10].copy()
df.columns = raw_df.iloc[header_row, 4:10].tolist()

# Clean column names
df.columns = [str(col).strip() for col in df.columns]

# Drop fully empty rows
df = df.dropna(how="all").reset_index(drop=True)

print("Extracted table shape:", df.shape)
display(df.head(10))
print(df.columns.tolist())

Extracted table shape: (110, 6)


,DATE,TRANSACTION,FROM/TO,AMOUNT,NOTE,BALANCE
0,2023-01-01 00:00:00,Income,MyCompany,4200,Salary - December 2022,5200
1,2023-01-02 00:00:00,Housing,Vavilon LTD,-105,Home maintenance,5095
2,2023-01-03 00:00:00,Transportation,Parkings,-10,Parking,5085
3,2023-01-04 00:00:00,Food,McDonalds,-40,NaN,5045
4,2023-01-05 00:00:00,Utilities,Water Co,-102,Water,4943
5,2023-01-06 00:00:00,Insurance,BeSafe LTD,-350,Life Insurance,4593
6,2023-01-07 00:00:00,Medical & Healthcare,St.Johns Hospital,-500,NaN,4093
7,2023-01-08 00:00:00,Saving & Investing,Great Bank,-220,NaN,3873
8,2023-01-09 00:00:00,Personal Spending,Best Gym,-55,Gym Membership,3818
9,2023-01-10 00:00:00,Recreation & Entertainment,NY Nicks,-182.4,Basketball Tickets,3635.6


['DATE', 'TRANSACTION', 'FROM/TO', 'AMOUNT', 'NOTE', 'BALANCE']


In [46]:
#######CLEAN AND CONVERT DATA TYPES #####

# Convert date column to datetime
df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")

# Convert numeric columns
df["AMOUNT"] = pd.to_numeric(df["AMOUNT"], errors="coerce")
df["BALANCE"] = pd.to_numeric(df["BALANCE"], errors="coerce")

# Clean text columns
text_cols = ["TRANSACTION", "FROM/TO", "NOTE"]
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

# Drop rows with critical missing values
df = df.dropna(subset=["DATE", "AMOUNT", "BALANCE"]).reset_index(drop=True)

print("Cleaned dataset shape:", df.shape)
display(df.head(10))

Cleaned dataset shape: (110, 6)


,DATE,TRANSACTION,FROM/TO,AMOUNT,NOTE,BALANCE
0,2023-01-01,Income,MyCompany,4200.0,Salary - December 2022,5200.0
1,2023-01-02,Housing,Vavilon LTD,-105.0,Home maintenance,5095.0
2,2023-01-03,Transportation,Parkings,-10.0,Parking,5085.0
3,2023-01-04,Food,McDonalds,-40.0,nan,5045.0
4,2023-01-05,Utilities,Water Co,-102.0,Water,4943.0
5,2023-01-06,Insurance,BeSafe LTD,-350.0,Life Insurance,4593.0
6,2023-01-07,Medical & Healthcare,St.Johns Hospital,-500.0,nan,4093.0
7,2023-01-08,Saving & Investing,Great Bank,-220.0,nan,3873.0
8,2023-01-09,Personal Spending,Best Gym,-55.0,Gym Membership,3818.0
9,2023-01-10,Recreation & Entertainment,NY Nicks,-182.4,Basketball Tickets,3635.6


In [47]:
###### DUPLICATE CONTROL #######

# Check duplicates before removal
duplicate_count_before = df.duplicated(
    subset=["DATE", "AMOUNT", "TRANSACTION", "FROM/TO"]
).sum()

print("Duplicate count before removal:", duplicate_count_before)

# Remove duplicates based on key transaction fields
df = df.drop_duplicates(
    subset=["DATE", "AMOUNT", "TRANSACTION", "FROM/TO"]
).reset_index(drop=True)

# Check duplicates after removal
duplicate_count_after = df.duplicated(
    subset=["DATE", "AMOUNT", "TRANSACTION", "FROM/TO"]
).sum()

print("Duplicate count after removal:", duplicate_count_after)
print("Dataset shape after duplicate removal:", df.shape)

Duplicate count before removal: 0
Duplicate count after removal: 0
Dataset shape after duplicate removal: (110, 6)


In [48]:
# Create time-based features
df["YEAR"] = df["DATE"].dt.year
df["MONTH"] = df["DATE"].dt.month
df["MONTH_NAME"] = df["DATE"].dt.month_name()
df["DAY"] = df["DATE"].dt.day

# Create flow type
df["FLOW_TYPE"] = df["AMOUNT"].apply(lambda x: "Income" if x > 0 else "Expense")

# Create absolute amount
df["ABS_AMOUNT"] = df["AMOUNT"].abs()

display(df.head(10))

,DATE,TRANSACTION,FROM/TO,AMOUNT,NOTE,BALANCE,YEAR,MONTH,MONTH_NAME,DAY,FLOW_TYPE,ABS_AMOUNT
0,2023-01-01,Income,MyCompany,4200.0,Salary - December 2022,5200.0,2023,1,January,1,Income,4200.0
1,2023-01-02,Housing,Vavilon LTD,-105.0,Home maintenance,5095.0,2023,1,January,2,Expense,105.0
2,2023-01-03,Transportation,Parkings,-10.0,Parking,5085.0,2023,1,January,3,Expense,10.0
3,2023-01-04,Food,McDonalds,-40.0,nan,5045.0,2023,1,January,4,Expense,40.0
4,2023-01-05,Utilities,Water Co,-102.0,Water,4943.0,2023,1,January,5,Expense,102.0
5,2023-01-06,Insurance,BeSafe LTD,-350.0,Life Insurance,4593.0,2023,1,January,6,Expense,350.0
6,2023-01-07,Medical & Healthcare,St.Johns Hospital,-500.0,nan,4093.0,2023,1,January,7,Expense,500.0
7,2023-01-08,Saving & Investing,Great Bank,-220.0,nan,3873.0,2023,1,January,8,Expense,220.0
8,2023-01-09,Personal Spending,Best Gym,-55.0,Gym Membership,3818.0,2023,1,January,9,Expense,55.0
9,2023-01-10,Recreation & Entertainment,NY Nicks,-182.4,Basketball Tickets,3635.6,2023,1,January,10,Expense,182.4


In [49]:
##### CATEGORY MAPPING ######

# transaction names into reporting categories
category_map = {
    "food": "Food & Beverage",
    "housing": "Housing",
    "income": "Income",
    "insurance": "Insurance",
    "medical healthcare": "Healthcare",
    "miscellaneous": "Other",
    "personal spending": "Personal Spending",
    "recreation entertainment": "Entertainment",
    "saving investing": "Saving & Investing",
    "transportation": "Transport",
    "utilities": "Utilities"
}

# Map category names
df["CATEGORY"] = df["TRANSACTION"].str.lower().map(category_map)

# Fill unmapped values if any
df["CATEGORY"] = df["CATEGORY"].fillna(df["TRANSACTION"].str.title())

display(df[["TRANSACTION", "CATEGORY"]].drop_duplicates().sort_values("TRANSACTION"))

,TRANSACTION,CATEGORY
3,Food,Food & Beverage
1,Housing,Housing
0,Income,Income
5,Insurance,Insurance
6,Medical & Healthcare,Medical & Healthcare
10,Miscellaneous,Other
8,Personal Spending,Personal Spending
9,Recreation & Entertainment,Recreation & Entertainment
7,Saving & Investing,Saving & Investing
2,Transportation,Transport


In [54]:
# Prepare selectable values for widgets
all_categories = sorted(df["CATEGORY"].dropna().unique().tolist())
all_companies = sorted(df["FROM/TO"].dropna().unique().tolist())
all_flow_types = sorted(df["FLOW_TYPE"].dropna().unique().tolist())
all_dates = sorted(df["DATE"].dt.date.unique())

In [55]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.express as px

# Date range selector
date_range = widgets.SelectionRangeSlider(
    options=all_dates,
    index=(0, len(all_dates) - 1),
    description="Date Range",
    layout=widgets.Layout(width="95%"),
    style={"description_width": "initial"}
)

# Category selector
category_select = widgets.SelectMultiple(
    options=all_categories,
    value=tuple(all_categories),
    description="Category",
    layout=widgets.Layout(width="300px", height="180px"),
    style={"description_width": "initial"}
)

# Company selector
company_select = widgets.SelectMultiple(
    options=all_companies,
    value=tuple(all_companies),
    description="Company",
    layout=widgets.Layout(width="300px", height="180px"),
    style={"description_width": "initial"}
)

# Flow type selector
flow_type_dropdown = widgets.Dropdown(
    options=["All", "Expense", "Income"],
    value="Expense",
    description="Flow Type",
    style={"description_width": "initial"}
)

# Pie chart grouping selector
group_by_dropdown = widgets.Dropdown(
    options=["CATEGORY", "FROM/TO"],
    value="CATEGORY",
    description="Pie Group",
    style={"description_width": "initial"}
)

# Sort column selector
sort_by_dropdown = widgets.Dropdown(
    options=["DATE", "AMOUNT", "ABS_AMOUNT", "BALANCE", "CATEGORY", "FROM/TO"],
    value="ABS_AMOUNT",
    description="Sort By",
    style={"description_width": "initial"}
)

# Sort order selector
ascending_toggle = widgets.ToggleButtons(
    options=[("Descending", False), ("Ascending", True)],
    value=False,
    description="Order",
    style={"description_width": "initial"}
)

# Top N selector for pie chart
top_n_slider = widgets.IntSlider(
    value=10,
    min=3,
    max=30,
    step=1,
    description="Top N",
    continuous_update=False,
    style={"description_width": "initial"}
)

# Max rows for displayed table
max_rows_slider = widgets.IntSlider(
    value=50,
    min=10,
    max=200,
    step=10,
    description="Table Rows",
    continuous_update=False,
    style={"description_width": "initial"}
)

In [56]:
summary_output = widgets.Output()
table_output = widgets.Output()
chart_output = widgets.Output()

In [61]:
def update_dashboard(change=None):
    with summary_output:
        clear_output(wait=True)
    with table_output:
        clear_output(wait=True)
    with chart_output:
        clear_output(wait=True)

    # Start from full dataframe
    filtered_df = df.copy()

    # Apply date filter
    start_date, end_date = date_range.value
    filtered_df = filtered_df[
        (filtered_df["DATE"].dt.date >= start_date) &
        (filtered_df["DATE"].dt.date <= end_date)
    ]

    # Apply category filter
    selected_categories = list(category_select.value)
    if selected_categories:
        filtered_df = filtered_df[filtered_df["CATEGORY"].isin(selected_categories)]

    # Apply company filter
    selected_companies = list(company_select.value)
    if selected_companies:
        filtered_df = filtered_df[filtered_df["FROM/TO"].isin(selected_companies)]

    # Apply flow type filter
    selected_flow = flow_type_dropdown.value
    if selected_flow != "All":
        filtered_df = filtered_df[filtered_df["FLOW_TYPE"] == selected_flow]

    # Sort table
    sort_col = sort_by_dropdown.value
    asc = ascending_toggle.value
    filtered_df = filtered_df.sort_values(by=sort_col, ascending=asc)

    # Summary section
    with summary_output:
        total_rows = len(filtered_df)
        total_absolute = filtered_df["ABS_AMOUNT"].sum()
        total_net = filtered_df["AMOUNT"].sum()

        print(f"Rows: {total_rows}")
        print(f"Total absolute amount: {total_absolute:,.2f}")
        print(f"Net amount: {total_net:,.2f}")

        if len(filtered_df) > 0:
            top_company_local = (
                filtered_df.groupby("FROM/TO")["ABS_AMOUNT"]
                .sum()
                .sort_values(ascending=False)
            )
            if len(top_company_local) > 0:
                print(f"Top company: {top_company_local.index[0]} ({top_company_local.iloc[0]:,.2f})")

            top_category_local = (
                filtered_df.groupby("CATEGORY")["ABS_AMOUNT"]
                .sum()
                .sort_values(ascending=False)
            )
            if len(top_category_local) > 0:
                print(f"Top category: {top_category_local.index[0]} ({top_category_local.iloc[0]:,.2f})")

    # Interactive table section
    with table_output:
        display(filtered_df.head(max_rows_slider.value))

    # Pie chart section
    with chart_output:
        if len(filtered_df) == 0:
            print("No data available for selected filters.")
            return

        group_col = group_by_dropdown.value

        grouped = (
            filtered_df
            .groupby(group_col)["ABS_AMOUNT"]
            .sum()
            .sort_values(ascending=False)
        )

        top_n = top_n_slider.value
        if len(grouped) > top_n:
            grouped_top = grouped.head(top_n).copy()
            other_sum = grouped.iloc[top_n:].sum()
            if other_sum > 0:
                grouped_top.loc["Other"] = other_sum
            grouped = grouped_top

        pie_df = grouped.reset_index()
        pie_df.columns = [group_col, "AMOUNT"]

        fig = px.pie(
            pie_df,
            names=group_col,
            values="AMOUNT",
            title=f"Distribution by {group_col}",
            width = 600,
            height = 600
        )

        fig.update_traces(
            textinfo="label+percent+value",
            texttemplate="%{label}<br>%{value:,.0f}<br>%{percent}"
        )

        fig.show()

In [62]:
controls = [
    date_range,
    category_select,
    company_select,
    flow_type_dropdown,
    group_by_dropdown,
    sort_by_dropdown,
    ascending_toggle,
    top_n_slider,
    max_rows_slider
]

for control in controls:
    control.observe(update_dashboard, names="value")

In [63]:
left_panel = widgets.VBox([
    category_select,
    company_select
])

right_panel = widgets.VBox([
    flow_type_dropdown,
    group_by_dropdown,
    sort_by_dropdown,
    ascending_toggle,
    top_n_slider,
    max_rows_slider
])

dashboard_ui = widgets.VBox([
    date_range,
    widgets.HBox([left_panel, right_panel]),
    summary_output,
    table_output,
    chart_output
])

display(dashboard_ui)
update_dashboard()